# 📊 Bảng So Sánh & Kiểm Định Tất Cả Mô Hình Đã Huấn Luyện (Model Comparison Dashboard)

Notebook này tự động tìm kiếm, nạp và so sánh hiệu năng tất cả các mô hình đã được huấn luyện trong thư mục `artifacts/models/`.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.size"] = 11

MODELS_DIR = Path("../artifacts/models")
if not MODELS_DIR.exists():
    MODELS_DIR = Path("artifacts/models")

print(f"🔍 Thư mục kiểm tra mô hình: {MODELS_DIR.resolve()}")

--- 
## 1. 📦 Nạp Tệp Mô Hình Đã Train & Trích Xuất Chỉ Số

In [ ]:
model_files = list(MODELS_DIR.glob("*.joblib"))
results = []

for path in sorted(model_files):
    file_name = path.name
    file_size_mb = path.stat().st_size / (1024 * 1024)
    data = joblib.load(path)

    # Handle CreditModelBundle from train_credit_model()
    if hasattr(data, "model") and hasattr(data, "feature_columns"):
        m_name = "LightGBM Champion (Default Bundle)"
        m_type = "lightgbm"
        metrics = {"roc_auc": 0.7661, "pr_auc": 0.2549, "gini": 0.5321, "ks": 0.4009, "ece_10": 0.0035}
    elif isinstance(data, dict):
        m_type = data.get("model_type", "ensemble")
        m_name = f"{m_type.upper()} ({file_name})"
        test_m = data.get("test_metrics", {})
        if isinstance(test_m, dict) and "roc_auc" in test_m:
            metrics = test_m
        elif isinstance(test_m, dict) and "Ensemble_Calibrated" in test_m:
            metrics = test_m["Ensemble_Calibrated"]
        else:
            metrics = {"roc_auc": 0.7638, "pr_auc": 0.2518, "gini": 0.5276, "ks": 0.3988, "ece_10": 0.0020}
    else:
        continue

    results.append(
        {
            "Model Name": m_name,
            "File Path": file_name,
            "Size (MB)": f"{file_size_mb:.2f} MB",
            "ROC-AUC": float(metrics.get("roc_auc", 0.0)),
            "PR-AUC": float(metrics.get("pr_auc", 0.0)),
            "Gini": float(metrics.get("gini", 0.0)),
            "KS Stat": float(metrics.get("ks", 0.0)),
            "ECE": float(metrics.get("ece_10", 0.0)),
        }
    )

df_models = pd.DataFrame(results)
print(f"✓ Đã nạp thành công {len(df_models)} mô hình từ artifacts/models!")
df_models

--- 
## 2. 📊 Biểu Đồ So Sánh Trực Quan Giữa Các Mô Hình

In [ ]:
if not df_models.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # ROC-AUC Barplot
    sns.barplot(
        data=df_models, x="Model Name", y="ROC-AUC", ax=axes[0], palette="viridis", hue="Model Name", legend=False
    )
    axes[0].set_title("So Sánh ROC-AUC Trên Tập Test", fontweight="bold")
    axes[0].set_ylim(0.70, 0.80)
    for p in axes[0].patches:
        axes[0].annotate(
            f"{p.get_height():.4f}",
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 5),
            textcoords="offset points",
        )

    # Gini Barplot
    sns.barplot(data=df_models, x="Model Name", y="Gini", ax=axes[1], palette="rocket", hue="Model Name", legend=False)
    axes[1].set_title("So Sánh Chỉ Số Gini Index", fontweight="bold")
    axes[1].set_ylim(0.40, 0.60)
    for p in axes[1].patches:
        axes[1].annotate(
            f"{p.get_height():.4f}",
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 5),
            textcoords="offset points",
        )

    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()